# Module 06 — Java GC & Gluten

In [ ]:
import sys
import os
sys.path.insert(0, '/workspace' if __import__('pathlib').Path('/workspace').exists() else '.')
from shared.spark_session import begin_module, ui_links
from shared import config
from shared.utils import timed_action

spark = begin_module('module-06-java-gc')
ui_links()  # App UI at :4040 is live after this cell

## Part A — JVM / GC

In [ ]:
trips = spark.read.parquet(*config.yellow_taxi_paths())
heavy = trips.groupBy('PULocationID', 'DOLocationID', 'VendorID').count()
_, t = timed_action(heavy)
print(f'Aggregation: {t:.2f}s — check GC Time in Executors tab')

In [ ]:
for item in ['spark.driver.memory', 'spark.executor.memory', 'spark.memory.fraction']:
    print(f'{item}: {spark.conf.get(item, "default")}')

## Part B — Gluten (Optional)

In [ ]:
GLUTEN_ENABLED = os.environ.get('SPARK_GLUTEN_ENABLED', 'false').lower() == 'true'
print(f'Gluten enabled: {GLUTEN_ENABLED}')
if GLUTEN_ENABLED:
    bench = spark.read.parquet(*config.yellow_taxi_paths())
    result = bench.groupBy('PULocationID').agg({'total_amount': 'sum'})
    result.explain('formatted')
    _, t = timed_action(result)
    print(f'Gluten benchmark: {t:.2f}s')

## Key Takeaways

1. Monitor GC Time before tuning JVM flags.
2. Gluten accelerates CPU-bound SQL but excludes Iceberg today.